# Audio Transcribing

In [ ]:
# ! pip install whisper # uncomment this to install whisper via pip
# ! pip install sounddevice # ucomment this to install sounddevice via pip
# ! pip install soundfile # ucomment this to install soundfile via pip
# ! pip install deep_translator # ucomment this to install deep_translator via pip
! uv add openai-whisper
! uv add sounddevice
! uv add soundfile
! uv add deep_translator

In [ ]:
# import
import os
import sys
import threading
import time

import sounddevice as sd
import soundfile as sf
import whisper

In [ ]:
os.environ["PATH"] += os.pathsep + r"/opt/homebrew/bin/ffmpeg"

In [ ]:
def show_timer(duration: int = 10) -> None:
    for i in range(duration):
        sys.stdout.write(f"\r⏳ Recording... {i + 1} seconds")
        sys.stdout.flush()
        time.sleep(1)
    print("\nRecording complete.")

In [ ]:
def record_audio(
    duration: int = 10, samplerate: int = 44100, temp_filename: str = "audio_temp.wav"
) -> str:
    print(f"🎤 Starting {duration}-second recording...")
    timer_thread = threading.Thread(target=show_timer, args=(duration,))
    timer_thread.start()

    audio = sd.rec(
        int(duration * samplerate), samplerate=samplerate, channels=1, dtype="int16"
    )
    sd.wait()
    timer_thread.join()

    sf.write(file=temp_filename, data=audio, samplerate=samplerate)
    print(f"\n✅ Saved recording: {temp_filename}")
    return temp_filename

In [ ]:
def transcribe_to_english(audio_path):
    print("🧠 Transcribing with Whisper...")
    model = whisper.load_model("base")
    result = model.transcribe(audio_path, fp16=False)
    return result["text"]

In [ ]:
if __name__ == "__main__":
    try:
        seconds = int(input("For how many seconds do you want to record? "))
        audio_file = record_audio(seconds)
        english_text = transcribe_to_english(audio_file)
        print("\n📜 English Transcript:\n", english_text)
    except ValueError:
        print("Please enter a valid number.")
    except Exception as e:
        print(f"Something went wrong {e}")